#### Import Packages

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import pandas as pd

In [ ]:
file_path = "cleaned_df_4lakh.csv"
cleaned_df = pd.read_csv(file_path)
print("flat_df loaded successfully.")
print("First 5 rows:")
display(cleaned_df.tail())

In [ ]:
X_cols = ['lon', 'lat', 'elevation', 'ph', 'organic_matter (%)',
          'total_nitrogen (%)', 'potassium (kg/ha)', 'p2o5 (kg/ha)',
          'boron (ppm)', 'zinc (ppm)', 'sand (%)', 'clay (%)', 'slit (%)',
          'parentsoil']

In [ ]:
Y_cols = ['Maize_OPV_MOP', 'Maize_OPV_DAP', 'Maize_OPV_UREA1', 'Maize_OPV_UREA2',
          'Maize_OPV_UREA3', 'Maize_OPV_organic', 'Maize_OPV_zinc', 'Maize_OPV_boron',
          'Maize_Hybrid_MOP', 'Maize_Hybrid_DAP', 'Maize_Hybrid_UREA1',
          'Maize_Hybrid_UREA2', 'Maize_Hybrid_UREA3', 'Maize_Hybrid_organic',
          'Maize_Hybrid_zinc', 'Maize_Hybrid_boron', 'Rice_OPV_MOP',
          'Rice_OPV_DAP', 'Rice_OPV_UREA1', 'Rice_OPV_UREA2', 'Rice_OPV_UREA3',
          'Rice_OPV_organic', 'Rice_OPV_zinc', 'Rice_OPV_boron', 'Rice_Hybrid_MOP',
          'Rice_Hybrid_DAP', 'Rice_Hybrid_UREA1', 'Rice_Hybrid_UREA2',
          'Rice_Hybrid_UREA3', 'Rice_Hybrid_organic', 'Rice_Hybrid_zinc',
          'Rice_Hybrid_boron', 'Wheat_OPV_MOP', 'Wheat_OPV_DAP', 'Wheat_OPV_UREA1',
          'Wheat_OPV_UREA2', 'Wheat_OPV_UREA3', 'Wheat_OPV_organic',
          'Wheat_OPV_zinc', 'Wheat_OPV_boron']

In [ ]:
X = cleaned_df[X_cols]
Y = cleaned_df[Y_cols]

#### Training

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, Y, test_size=0.2, random_state=42)

In [ ]:
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.multioutput import MultiOutputRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

models = {
    "Linear Regression": MultiOutputRegressor(
        LinearRegression()
    ),

    "Ridge Regression": MultiOutputRegressor(
        Ridge(alpha=1)
    ),

    "Random Forest": MultiOutputRegressor(
        RandomForestRegressor(
            n_estimators=200,
            min_samples_split=5,
            max_depth=None,
            random_state=42,
            n_jobs=-1
        )
    ),

    "XGBoost": MultiOutputRegressor(
        XGBRegressor(
            n_estimators=500,
            learning_rate=0.1,
            max_depth=7,
            subsample=0.8,
            random_state=42,
            n_jobs=-1
        )
    ),

    "LightGBM": MultiOutputRegressor(
        LGBMRegressor(
            n_estimators=500,
            learning_rate=0.1,
            num_leaves=31,
            min_data_in_leaf=20,
            random_state=42,
            n_jobs=-1,
            verbose=-1
        )
    ),
}

##### Train and evaluate each model ---

In [ ]:
from sklearn.pipeline import Pipeline

In [ ]:
import warnings
import logging

In [ ]:
warnings.filterwarnings("ignore")

# LightGBM prints warnings through its own logger/stdout.
# This usually suppresses most python-side noise:
logging.getLogger("lightgbm").setLevel(logging.ERROR)

In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

# Crop groups
crop_groups = {
    "Maize":  [c for c in Y_cols if c.startswith("Maize")],
    "Rice":   [c for c in Y_cols if c.startswith("Rice")],
    "Wheat":  [c for c in Y_cols if c.startswith("Wheat")]
}

# Fertilizer groups
fert_groups = {}
for c in Y_cols:
    fert = c.split("_")[-1]
    fert_groups.setdefault(fert, []).append(c)

y_idx = {c: i for i, c in enumerate(Y_cols)}

def compute_metrics(y_true, y_pred, cols):
    idx = [y_idx[c] for c in cols]
    yt = y_true.iloc[:, idx]
    yp = y_pred[:, idx]
    return (
        r2_score(yt, yp, multioutput="uniform_average"),
        mean_absolute_error(yt, yp),
        mean_squared_error(yt, yp)  # RMSE
    )


In [ ]:
import time

global_results = {}
crop_results = {}
fert_results = {}

for name, model in models.items():

    print(f"\n🚀 Starting model: {name}")
    start_model = time.time()

    pipe = Pipeline(steps=[("model", model)])

    print("  ▶ Fitting...")
    t0 = time.time()
    pipe.fit(X_train, y_train)
    print(f"  ✅ Fit done in {time.time() - t0:.2f}s")

    print("  ▶ Predicting...")
    t0 = time.time()
    y_pred = pipe.predict(X_test)
    print(f"  ✅ Predict done in {time.time() - t0:.2f}s")

    # ---- Global ----
    print("  ▶ Computing global metrics...")
    global_results[name] = {
        "R²": r2_score(y_test, y_pred, multioutput="uniform_average"),
        "MAE": mean_absolute_error(y_test, y_pred),
        "RMSE": mean_squared_error(y_test, y_pred)
    }

    # ---- Crop-wise ----
    print("  ▶ Crop-wise metrics...")
    crop_results[name] = {}
    for crop, cols in crop_groups.items():
        r2, _, _ = compute_metrics(y_test, y_pred, cols)
        crop_results[name][crop] = r2

    # ---- Fertilizer-wise ----
    print("  ▶ Fertilizer-wise metrics...")
    fert_results[name] = {}
    for fert, cols in fert_groups.items():
        r2, _, _ = compute_metrics(y_test, y_pred, cols)
        fert_results[name][fert] = r2

    print(f"🏁 Finished {name} in {time.time() - start_model:.2f}s")

In [ ]:
global_df = pd.DataFrame(global_results).T.sort_values("R²", ascending=False)
print("\n=== Global Model Comparison ===")
print(global_df.round(3))

In [ ]:
crop_df = pd.DataFrame(crop_results).T
print("\n=== Model-wise Crop Performance (R²) ===")
print(crop_df.round(3))

In [ ]:
fert_df = pd.DataFrame(fert_results).T
print("\n=== Model-wise Fertilizer Performance (R²) ===")
print(fert_df.round(3))

In [ ]:
from sklearn.model_selection import KFold
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.pipeline import Pipeline
import numpy as np

# Initialize KFold for cross-validation
kf = KFold(n_splits=5, shuffle=True, random_state=42)

cv_results = {}

print("\n--- Cross-validation Results (10 Folds) ---")

for name, model in models.items():
    r2_scores = []
    mae_scores = []
    rmse_scores = []

    # Create a pipeline for the current model
    pipe = Pipeline(steps=[("model", model)])

    # Perform cross-validation
    for train_index, val_index in kf.split(X, Y):
        X_train_fold, X_val_fold = X.iloc[train_index], X.iloc[val_index]
        y_train_fold, y_val_fold = Y.iloc[train_index], Y.iloc[val_index]

        pipe.fit(X_train_fold, y_train_fold)
        y_pred_fold = pipe.predict(X_val_fold)

        r2_scores.append(r2_score(y_val_fold, y_pred_fold, multioutput='uniform_average'))
        mae_scores.append(mean_absolute_error(y_val_fold, y_pred_fold))
        rmse_scores.append(mean_squared_error(y_val_fold, y_pred_fold))

    # Calculate and store average metrics
    avg_r2 = np.mean(r2_scores)
    avg_mae = np.mean(mae_scores)
    avg_rmse = np.mean(rmse_scores)

    cv_results[name] = {"R²": avg_r2, "MAE": avg_mae, "RMSE": avg_rmse}

    print(f"\n🔹 {name}")
    print(f"Average R²:   {avg_r2:.3f}")
    print(f"Average MAE:  {avg_mae:.3f}")
    print(f"Average RMSE: {avg_rmse:.3f}")
